In [ ]:
import pandas as pd
import numpy as np
import textstat
import spacy
import nltk
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

from langchain_openai import ChatOpenAI
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from dotenv import load_dotenv
load_dotenv()

input_csv = "evaluation_details/evaluation_pdfs/full_dateset_pdf.csv"
df = pd.read_csv(input_csv)
questions = df["Question"]
sample_queries = questions.tolist()
answers = df["Answer"]
expected_responses = answers.tolist()

/Users/ukostuch/Documents/THESIS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
questions.head()

0    How does the Gated PixelCNN address the blind ...
1    How can Conditional PixelCNN be used as an ima...
2    What performance did Gated PixelCNN achieve on...
3    What is the main advantage of incorporating eq...
4    How does Equivariant MuZero handle the C4 symm...
Name: Question, dtype: object

In [3]:
answers.head()

0    It uses two convolutional network stacks: a ve...
1    By replacing the deconvolutional decoder with ...
2    On 32×32 ImageNet, Gated PixelCNN achieved 3.8...
3    By making the neural networks equivariant to s...
4    The latent state z consists of 4 equally shape...
Name: Answer, dtype: object

In [ ]:
%run agentic_rag_chatbot.ipynb


Added SUMMARY chunk:
Title: Playing Atari with Deep Reinforcement Learning
Authors: Volodymyr Mnih; Koray Kavukcuoglu; David Silver; Alex Graves; Ioannis Antonoglou; Daan Wierstra; Martin Riedmiller
Abstract: We present the first deep learning model to successfully learn control policies directly from high-dimensional sensory input using reinforcement learning. The model is a convolutional neural network, trained with ...
Metadata: {'source': '1312.5602v1.pdf', 'doc_id': '1312.5602v1', 'title': 'Playing Atari with Deep Reinforcement Learning', 'authors': 'Volodymyr Mnih; Koray Kavukcuoglu; David Silver; Alex Graves; Ioannis Antonoglou; Daan Wierstra; Martin Riedmiller', 'pub_date': '2013-12-19', 'type': 'summary'}

Added SECTION chunk:
Introduction
Learning to control agents directly from high-dimensional sensory inputs like vision and speech is one of the long-standing challenges of reinforcement learning (RL). Most successful RL applications that operate on these domains have relied

In [5]:
given_answers = []
for query in sample_queries:
    answer = ask_chatbot(query)
    given_answers.append(answer)

Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...
Fetching from PDFs...


In [6]:
print(len(given_answers))

30


In [7]:
qa_df = pd.DataFrame({
    "question": sample_queries,
    "expected_response": expected_responses,
    "given_answer": given_answers
})
qa_df.to_csv("qa_results.csv", index=False)

In [8]:
dataset = []
for query, reference, response in zip(sample_queries, expected_responses, given_answers):
    retrieved_contexts = get_retrieved_contexts(query, source_key=os.getenv("SOURCE_KEY"), k=5)
    dataset.append({
        "user_input": query,
        "retrieved_contexts": retrieved_contexts,
        "response": response,
        "reference": reference,
    })
evaluation_dataset = EvaluationDataset.from_list(dataset)

eval_llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model=os.getenv("EVAL_MODEL_NAME")
)


In [9]:
model_name = os.getenv("MODEL_NAME")
model = model_name.split("/", 1)[1]

eval_model_name = os.getenv("EVAL_MODEL_NAME")
eval_model = eval_model_name.split("/", 1)[1]

evaluator_llm = LangchainLLMWrapper(eval_llm)
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
)
df_metrics = result.to_pandas()
df_metrics.to_csv(f"details_ragas_{model}_{eval_model}.csv", index=False)

Evaluating: 100%|██████████| 90/90 [01:55<00:00,  1.29s/it]


In [ ]:
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nlp = spacy.load("en_core_web_sm")

def compute_extra_metrics(texts, embeddings_model):
    if isinstance(texts, str):
        texts = [texts]
    results = []
    for text in texts:
        doc = nlp(text)
        words = [token.text for token in doc if token.is_alpha]
        sentences = list(doc.sents)
        avg_word_len = np.mean([len(w) for w in words]) if words else 0
        avg_sent_len = np.mean([len(s) for s in sentences]) if sentences else 0
        syntactic_complexity = np.mean([len([t for t in s if t.dep_ == "ROOT"]) for s in sentences]) if sentences else 0
        flesch_reading_ease = textstat.flesch_reading_ease(text)
        flesch_kincaid_grade = textstat.flesch_kincaid_grade(text)
        smog_index = textstat.smog_index(text)
        gunning_fog = textstat.gunning_fog(text)
        ttr = len(set(words)) / len(words) if words else 0
        sent_texts = [s.text for s in sentences]
        sent_embeddings = embeddings_model.embed_documents(sent_texts) if len(sent_texts) > 1 else []
        if len(sent_embeddings) > 1:
            sims = cosine_similarity(sent_embeddings)
            avg_sim = (np.sum(sims) - np.trace(sims)) / (sims.shape[0]**2 - sims.shape[0])
            embedding_diversity = 1 - avg_sim
        else:
            embedding_diversity = 0
        bigrams = list(nltk.bigrams(words))
        bigram_diversity = len(set(bigrams)) / len(bigrams) if bigrams else 0
        reasoning_keywords = {"because", "therefore", "hence", "so", "since", "thus", "if", "then", "reason"}
        reasoning_ratio = len([w for w in words if w.lower() in reasoning_keywords]) / len(words) if words else 0
        reasoning_steps = sum([1 for w in words if w.lower() in {"if", "then", "so", "thus"}])
        depths = [max([len(list(t.ancestors)) for t in s]) for s in sentences] if sentences else [0]
        inference_complexity = np.mean(depths)
        results.append({
            "avg_word_length": avg_word_len,
            "avg_sentence_length": avg_sent_len,
            "syntactic_complexity": syntactic_complexity,
            "flesch_reading_ease": ,
            "flesch_kincaid_grade": flesch_kincaid_grade,
            "smog_index": smog_index,
            "gunning_fog": gunning_fog,
            "type_token_ratio": ttr,
            "embedding_diversity": embedding_diversity,
            "bigram_diversity": bigram_diversity,
            "reasoning_keyword_ratio": reasoning_ratio,
            "average_reasoning_steps": reasoning_steps,
            "inference_complexity_score": inference_complexity,
        })
    return results if len(results) > 1 else results[0]

extra_metrics_results = []
for i, ans in enumerate(given_answers):
    metrics = compute_extra_metrics(ans, embeddings)
    metrics["context_recall"] = df_metrics.loc[i, "context_recall"]
    metrics["faithfulness"] = df_metrics.loc[i, "faithfulness"]
    metrics["factual_correctness(mode=f1)"] = float(df_metrics.loc[i, "factual_correctness(mode=f1)"])
    extra_metrics_results.append(metrics)
df_extra = pd.DataFrame(extra_metrics_results)
df_extra.to_csv(f"metrics_{model}_{eval_model}.csv", index=False)

[nltk_data] Downloading package punkt to /Users/ukostuch/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/ukostuch/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [11]:
df_extra.head()

,avg_word_length,avg_sentence_length,syntactic_complexity,flesch_reading_ease,flesch_kincaid_grade,smog_index,gunning_fog,type_token_ratio,embedding_diversity,bigram_diversity,reasoning_keyword_ratio,average_reasoning_steps,inference_complexity_score,context_recall,faithfulness,factual_correctness(mode=f1)
0,5.085202,21.153846,1.0,39.809636,12.788162,14.937676,16.359596,0.520179,0.710526,0.837838,0.008969,2,4.769231,1.000000,0.950000,0.84
1,5.401786,27.727273,1.0,21.333939,14.838182,16.218646,18.787879,0.482143,0.564492,0.847534,0.000000,0,4.727273,1.000000,0.933333,0.31
2,5.500000,28.666667,1.0,24.767749,16.181039,18.243606,20.656277,0.608108,0.567963,0.863014,0.000000,0,5.666667,0.333333,0.666667,0.17
3,5.557522,32.750000,1.0,22.437356,17.147854,17.122413,19.441593,0.654867,0.542037,0.937500,0.008850,1,8.500000,0.500000,0.888889,0.17
4,5.234432,26.666667,1.0,40.881225,12.144852,14.265293,15.473309,0.571429,0.696600,0.900735,0.000000,0,5.000000,1.000000,1.000000,0.59


In [12]:
df_avg = pd.DataFrame(df_extra.mean(axis=0)).T
df_avg.insert(0, "model_name", model_name)
csv_path = "average_metrics.csv"
if os.path.exists(csv_path):
    df_avg.to_csv(csv_path, mode="a", header=False, index=False)
else:
    df_avg.to_csv(csv_path, index=False)
print("All metrics saved.")

All metrics saved.
